In [ ]:
# Step 1: Data Cleaning & Preprocessing
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# Load the dataset
df = pd.read_csv('Restaurant_orders.xlsx - Restaurant_orders.csv')

# Clean columns strictly following the syllabus
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Check for missing values
print("Missing Values Percentage:\n", df.isna().mean() * 100)

# Handle nulls
df = df.fillna(0)

# Convert dates to datetime objects
df['order_date_and_time'] = pd.to_datetime(df['order_date_and_time'], errors='coerce')


In [ ]:
# Step 2: Feature Engineering & Groupby Analysis
# Extract the day of the week using the new syllabus method
df['day_of_week'] = df['order_date_and_time'].dt.day_name()

# Calculate Net Revenue (Order Value minus Refunds)
df['net_revenue'] = df['order_value'] - df['refunds/chargebacks']

# Groupby analysis: Find which payment method generates the most total revenue
revenue_by_payment = df.groupby('payment_method')[['net_revenue']].sum().sort_values(by='net_revenue', ascending=False).head()
print("Top Revenue by Payment Method:\n", revenue_by_payment)


In [ ]:
# Step 3: Syllabus-Compliant Visualizations

# 1. Count of Payment Methods
plt.figure(figsize=(8, 4))
sns.countplot(data=df, x='payment_method')
plt.title("Distribution of Payment Methods")
plt.show()

# 2. Average Net Revenue by Day of the Week (strictly without estimator=sum or xticks)
plt.figure(figsize=(10, 4))
sns.barplot(data=df, x='day_of_week', y='net_revenue')
plt.title("Average Net Revenue by Day of Week")
plt.show()

# 3. Correlation Heatmap before Machine Learning
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap of Restaurant Orders Data")
plt.show()


In [ ]:
# Step 4: Predictive Modeling - Linear Regression
# We want to predict the continuous numerical 'order_value', so we use Linear Regression

le = LabelEncoder()
# Encode categorical text columns into numbers
df['payment_method_n'] = le.fit_transform(df['payment_method'].astype(str))
df['discounts_and_offers_n'] = le.fit_transform(df['discounts_and_offers'].astype(str))
df['day_of_week_n'] = le.fit_transform(df['day_of_week'].astype(str))

# Define X (Inputs) and y (Output Target)
X = df[['payment_method_n', 'discounts_and_offers_n', 'delivery_fee', 'payment_processing_fee', 'day_of_week_n']]
y = df['order_value']

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the model
model = LinearRegression()
model.fit(X_train, y_train)

# Score the model
print("Linear Regression Model Score (R-squared):", model.score(X_test, y_test))


In [ ]:
# Step 5: Model Prediction
# Dummy array corresponding to X: [payment_method_n, discounts_and_offers_n, delivery_fee, payment_processing_fee, day_of_week_n]
prediction = model.predict([[1, 2, 40.0, 15.0, 3]])
print("Predicted Order Value:", prediction)
